# Full-size red/white models

This is the all-red/white stage. Use all red and white reviews from saved `train.parquet` and the entire red/white saved validation partition. The earlier cells retain the team's explored count/bigram plus review-length models and their results. The final section uses the same TF-IDF settings and three classifiers as the 10-, 100-, and 1,000-review notebooks for a closer comparison. `test.parquet` stays reserved.

The saved split contains one red/white review whose text duplicates a validation review after whitespace normalization. This notebook removes its **training** copy and one additional within-training normalized duplicate and keeps the entire validation partition.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
included_types = ['red', 'white']
train = pd.read_parquet('train.parquet')
validation = pd.read_parquet('validation.parquet')
train = train.loc[train['color'].isin(included_types)].copy()
validation = validation.loc[validation['color'].isin(included_types)].copy()
assert set(included_types) <= set(train['color'])
assert set(included_types) <= set(validation['color'])
assert train.index.intersection(validation.index).empty, 'Train and validation rows overlap.'
assert train['description'].notna().all() and validation['description'].notna().all()
normalize_review = lambda s: s.str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
validation_keys = set(normalize_review(validation['description']))
train = train.loc[~normalize_review(train['description']).isin(validation_keys)].copy()
train = train.loc[~normalize_review(train['description']).duplicated()].copy()
assert set(normalize_review(train['description'])).isdisjoint(validation_keys)

wine = pd.concat([train, validation])  # Exploratory summaries only; never fit on this combined frame.
pd.DataFrame({
    'training': train['color'].value_counts(),
    'validation': validation['color'].value_counts(),
}).reindex(included_types)

## Review length and data size

Check lengths before vectorizing. The count matrix will be sparse, so memory scales with the number of words actually present in each review rather than rows × vocabulary size.

## Review length features

Count alphabetic words in each description and measure their mean and maximum length. Punctuation and numbers are excluded. Descriptions with no alphabetic words receive zeros. These features come only from `description`.


In [ ]:
import re

word_pattern = re.compile(r'(?u)\b[^\W\d_]+\b')
length_feature_cols = ['review_word_count', 'mean_word_length', 'max_word_length']

def review_length_features(description):
    lengths = [len(word) for word in word_pattern.findall(description)]
    if not lengths:
        return 0, 0.0, 0
    return len(lengths), sum(lengths) / len(lengths), max(lengths)

for frame in (train, validation):
    frame[length_feature_cols] = pd.DataFrame(
        [review_length_features(text) for text in frame['description'].fillna('')],
        index=frame.index,
        columns=length_feature_cols,
    )
wine = pd.concat([train, validation])
train[length_feature_cols].describe().round(1)

In [ ]:
review_chars = wine['description'].str.len()
print(f'{len(wine):,} reviews; {wine["description"].nunique():,} distinct descriptions')
print(f'Raw description text: {wine["description"].memory_usage(deep=True) / 2**20:,.1f} MiB')
pd.DataFrame({'words': wine['review_word_count'], 'characters': review_chars}).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

In [ ]:
wine.columns

## Use the saved split

`data_split.ipynb` already made a stratified 60/20/20 train/validation/test split. The notebook uses its saved training and validation rows, including all colors in `included_types`. Fit the vocabulary and feature transformations on training reviews only. Validation measures generalization during model comparison; the test split is reserved for final evaluation.

In [ ]:
y_train = train['color']
y_val = validation['color']
print(f'Train: {len(train):,}; validation: {len(validation):,}')
print('Training class proportions:')
print(y_train.value_counts(normalize=True).reindex(included_types).round(3))

## Unigram and bigram bag of words

`CountVectorizer` counts individual words and adjacent word pairs. `min_df=5` omits very rare terms, and `max_features=50_000` caps the vocabulary. The tokenizer keeps alphabetic words of at least two characters. The vocabulary is fit on training reviews and reused for validation reviews.

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5, max_features=50_000, token_pattern=r'(?u)\b[^\W\d_]{2,}\b', dtype=np.int32)
X_train_text = vectorizer.fit_transform(train['description'].fillna(''))
X_val_text = vectorizer.transform(validation['description'].fillna(''))

def sparse_mib(matrix):
    return (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 2**20

print(f'Vocabulary: {len(vectorizer.vocabulary_):,} unigrams/bigrams')
print(f'Train matrix: {X_train_text.shape}, {X_train_text.nnz:,} nonzero counts, {sparse_mib(X_train_text):,.1f} MiB')
print(f'Validation matrix: {X_val_text.shape}, {X_val_text.nnz:,} nonzero counts, {sparse_mib(X_val_text):,.1f} MiB')
print('Sample features:', vectorizer.get_feature_names_out()[:20].tolist())

## Other available predictors

The remaining columns can be added later. Numeric columns need imputation; categorical columns need missing-value handling and encoding. Check cardinality before one-hot encoding `winery`, `designation`, or `title`, since these may create large feature spaces. `title` and `designation` can also contain grape or color names, so decide whether those clues are allowed in the eventual prediction setting. `points` and `price` may be unavailable at prediction time.

In [ ]:
other_columns = [c for c in wine.columns if c not in ['description', 'color', *length_feature_cols]]
pd.DataFrame({
    'dtype': wine[other_columns].dtypes.astype(str),
    'missing': wine[other_columns].isna().sum(),
    'unique_non_null': wine[other_columns].nunique(dropna=True),
}).sort_values('unique_non_null', ascending=False)

## Description-only classifiers and bias–variance comparison

A majority-class dummy model gives a reference for this imbalanced red/white problem. Compare it with multinomial Naive Bayes, logistic regression, and a linear SVM. All three text classifiers use unigram/bigram counts and three length features; logistic regression and SVM use TF-IDF weights. Fit scaling and TF-IDF on training data only.

Compare **training versus validation macro F1** and balanced accuracy. Low scores on both splits suggest high bias; a strong training score with a much lower validation score suggests high variance. These are diagnostic clues, not formal estimates of statistical bias or variance. Macro F1 gives rare colors equal weight, while accuracy can hide poor minority-class performance.

In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

length_scaler = MaxAbsScaler()
X_train_length = csr_matrix(length_scaler.fit_transform(train[length_feature_cols]))
X_val_length = csr_matrix(length_scaler.transform(validation[length_feature_cols]))

tfidf = TfidfTransformer(sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

X_train_counts_with_length = hstack([X_train_text, X_train_length], format='csr')
X_val_counts_with_length = hstack([X_val_text, X_val_length], format='csr')
X_train_tfidf_with_length = hstack([X_train_tfidf, X_train_length], format='csr')
X_val_tfidf_with_length = hstack([X_val_tfidf, X_val_length], format='csr')
print(f'Train matrix with review length features: {X_train_tfidf_with_length.shape}, {sparse_mib(X_train_tfidf_with_length):,.1f} MiB')

In [ ]:
models = {
    'Majority class': (DummyClassifier(strategy='most_frequent'), X_train_counts_with_length, X_val_counts_with_length),
    'Multinomial Naive Bayes': (MultinomialNB(alpha=1.0), X_train_counts_with_length, X_val_counts_with_length),
    'Logistic regression': (LogisticRegression(max_iter=1000, solver='lbfgs'), X_train_tfidf_with_length, X_val_tfidf_with_length),
    'Linear SVM': (LinearSVC(max_iter=5000, dual=False), X_train_tfidf_with_length, X_val_tfidf_with_length),
}

score_rows = []
predictions = {}
for name, (model, X_fit, X_eval) in models.items():
    model.fit(X_fit, y_train)
    for split, X, y in [('training', X_fit, y_train), ('validation', X_eval, y_val)]:
        y_pred = model.predict(X)
        if split == 'validation':
            predictions[name] = y_pred
        score_rows.append({
            'model': name,
            'split': split,
            'accuracy': accuracy_score(y, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y, y_pred),
            'macro_f1': f1_score(y, y_pred, labels=included_types, average='macro', zero_division=0),
        })
    print(f'Finished: {name}')

scores = pd.DataFrame(score_rows)
results_df = scores.pivot(index='model', columns='split', values=['accuracy', 'balanced_accuracy', 'macro_f1'])
results_df.columns = [f'{metric}_{split}' for metric, split in results_df.columns]
results_df['macro_f1_gap'] = results_df['macro_f1_training'] - results_df['macro_f1_validation']
results_df['balanced_accuracy_gap'] = results_df['balanced_accuracy_training'] - results_df['balanced_accuracy_validation']
results_df = results_df.sort_values('macro_f1_validation', ascending=False)
print(results_df.round(3).to_string())

## Per-class validation errors

Inspect precision and recall for every color. The confusion-matrix rows are actual colors and columns are predicted colors. This reveals whether a good overall score hides weak performance on `rose`, `dessert`, or `sparkling`.

In [ ]:
for name, y_pred in predictions.items():
    print(f'\n{name}')
    print(classification_report(y_val, y_pred, labels=included_types, zero_division=0))
    print(pd.DataFrame(
        confusion_matrix(y_val, y_pred, labels=included_types),
        index=[f'actual {color}' for color in included_types],
        columns=[f'predicted {color}' for color in included_types],
    ))

## Misclassified validation reviews

Inspect errors from the model with the highest validation macro F1. `variety` appears only in this error table and is never a feature. The original row index is retained for lookup.

In [ ]:
model_to_inspect = results_df.index[0]  # Or choose a name from predictions.
y_pred = predictions[model_to_inspect]
wrong = y_pred != y_val.to_numpy()

error_df = validation.loc[y_val.index[wrong]].copy()
error_df = error_df.rename(columns={'color': 'actual_color'})
error_df.insert(1, 'predicted_color', y_pred[wrong])
first_cols = ['actual_color', 'predicted_color', 'variety', 'description', *length_feature_cols]
error_df = error_df[first_cols + [col for col in error_df.columns if col not in first_cols]]
print(f'{model_to_inspect}: {len(error_df):,} incorrect predictions out of {len(validation):,} validation reviews')
error_df

In [ ]:
error_df['actual_color'].value_counts().reindex(included_types, fill_value=0).to_frame('validation_errors')

In [ ]:
error_df['variety'].value_counts().head(20)

In [ ]:
error_df[['actual_color', 'predicted_color', 'variety', 'description']].head(10)

In [ ]:
# Compare the score gap with the validation score before choosing the next experiment.
diagnostic = results_df[['macro_f1_training', 'macro_f1_validation', 'macro_f1_gap',
                         'balanced_accuracy_training', 'balanced_accuracy_validation',
                         'balanced_accuracy_gap']].round(3)
print('A large positive gap suggests high variance; low scores on both splits suggest high bias.')
print('Next: test stronger regularization or more training data for models with large gaps,')
print('and class weighting or improved features for colors with low validation recall.')
print(diagnostic.to_string())

## Directly comparable full-size models

These use the same TF-IDF vocabulary settings and classifier settings as Models 1, 2, and 4. The full-size population also includes many more varieties and regions, so any score change combines training-size and task-scope effects. The planned intermediate stage is all California red and white varieties.

In [ ]:
from IPython.display import display
LABELS = ['red', 'white']
SEED = 42
assert set(train['color']) == set(validation['color']) == set(LABELS)
assert train.index.intersection(validation.index).empty
normalize = lambda s: s.str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
assert set(normalize(train['description'])).isdisjoint(normalize(validation['description']))
print(f'Full red/white training: {len(train):,}; validation: {len(validation):,}')
display(pd.DataFrame({'training': train['color'].value_counts(), 'validation': validation['color'].value_counts()}).reindex(LABELS))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10_000, sublinear_tf=True)

models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Naive Bayes': make_pipeline(text_features(), MultinomialNB(alpha=1.0)),
    'Logistic regression': make_pipeline(text_features(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)),
    'Linear SVM': make_pipeline(text_features(), LinearSVC(C=1.0, max_iter=5000, random_state=SEED)),
}
X_train, y_train = train['description'], train['color']
X_val, y_val = validation['description'], validation['color']
score_rows = []
validation_predictions = {}

def record_scores(name, train_true, train_pred, val_true, val_pred):
    for split, actual, predicted in [('training', train_true, train_pred), ('validation', val_true, val_pred)]:
        score_rows.append({'model': name, 'split': split, 'n': len(actual), 'accuracy': accuracy_score(actual, predicted), 'macro_f1': f1_score(actual, predicted, labels=LABELS, average='macro', zero_division=0)})
    validation_predictions[name] = np.asarray(val_pred)

for name, model in models.items():
    model.fit(X_train, y_train)
    record_scores(name, y_train, model.predict(X_train), y_val, model.predict(X_val))
    print('Finished', name)

In [ ]:
def show_comparison():
    scores = pd.DataFrame(score_rows).pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
    scores.columns = [f'{metric}_{split}' for metric, split in scores.columns]
    scores['accuracy_gap'] = scores['accuracy_training'] - scores['accuracy_validation']
    scores['macro_f1_gap'] = scores['macro_f1_training'] - scores['macro_f1_validation']
    display(scores.sort_values('macro_f1_validation', ascending=False).round(3))
    for name, predicted in validation_predictions.items():
        print(f'\n{name}: validation report')
        print(classification_report(y_val, predicted, labels=LABELS, zero_division=0, digits=3))
        display(pd.DataFrame(confusion_matrix(y_val, predicted, labels=LABELS), index=['actual red', 'actual white'], columns=['predicted red', 'predicted white']))
    return scores

comparison = show_comparison()